# 1.2 分布式 GMRES Profiling

## 本节学习目标

- 拆解 GMRES 阶段
- 理解 profile 字段与边界

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v msprof
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 阶段拆解

每个 Arnoldi 步包含：把局部向量放入全局 Device buffer → HCCL AllReduce 形成全局向量 → 局部 SpMV → Dot/Norm、正交化 AXPY（标量再经 AllReduce 规约）。Givens/Hessenberg 是小规模控制计算。

## Profiling 字段

工程记录 SpMV、Dot、AXPY、Norm、Givens、HCCL communication、ACL transfer、synchronization、AllReduce/AllGather calls。kernel launch 统计 Ascend C RTC kernel 的 Host 提交开销；正式路径中 SpMV/Dot/Norm/AXPY/Scale 均为真实 Device kernel，msprof 中应能看到对应的 kernel/task 时间。注意当前 Device 路径不使用 AllGather（全局向量经 Device placement + AllReduce 形成），其计数为 0。

## 预期现象与结果分析

这些阶段是应用级测量。SpMV/Dot/AXPY/Norm 由 Ascend C RTC Device kernel 执行，其 Device 侧耗时应结合 msprof timeline 读取；HCCL 与 ACL transfer 是实际 Device 通信/搬移路径。

## 课后实践

将一次 Arnoldi 迭代映射到 profile 字段和 collective。

参考答案见 `answer/01.02_answer.md`。

## 直接执行实验

该 Cell 建立单 Rank 基线。SpMV/Dot/Norm/AXPY/Scale 由 Ascend C RTC Device kernel 执行，HCCL 直接规约 Device buffer，Host 只处理小规模 Hessenberg/Givens 与最终校验。Profiling 小节再用相同参数执行 msprof。命令使用 --warmup 0：矩阵读取/分区与 communicator 初始化在 repeat 外只做一次；每次 solve 会重建 solver 内的 RTC/Device 状态，--warmup 只是额外执行并丢弃 N 次完整 solver invocation，不代表 warm cache 稳态指标，也不等于完整进程冷启动。


In [ ]:
%%bash
set -e
cd src/dis_gmres
DIS_GMRES_REQUIRE_REAL=1 bash scripts/build.sh
./build/bin/dis_gmres --matrix U2 --rank 0 --world-size 1 --device 0 --warmup 0 --repeat 3 --restart 30 --max-iterations 300 --tolerance 1e-6


## 采集并导出当前 Profile

下面使用与基线完全相同的参数。输出目录必须不存在，避免新旧采集文件混合。`msprof --application` 采集 Runtime/Task 时间，随后显式导出 Summary 与 Timeline。


In [ ]:
%%bash
set -euo pipefail
cd src/dis_gmres
PROFILE_DIR=profiling/u2_r1_baseline
if [[ -e "${PROFILE_DIR}" ]]; then echo "请先为本次实验选择新的 PROFILE_DIR" >&2; exit 1; fi
msprof --application="./build/bin/dis_gmres --matrix U2 --rank 0 --world-size 1 --device 0 --warmup 0 --repeat 3 --restart 30 --max-iterations 300 --tolerance 1e-6" --output="${PROFILE_DIR}" --runtime-api=on --task-time=on
msprof --export=on --output="${PROFILE_DIR}"
printf 'Profile exported to %s\n' "${PROFILE_DIR}"


## 校验 Profile 产物

该 Cell 只读取本次 `PROFILE_DIR` 的 CSV 产物，不读取源码。它按名称列累计耗时（ns 自动转 us）并输出 Top 10、总时长与占比；找不到名称/耗时列或没有可解析行时直接失败。


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

profile_dir = Path('src/dis_gmres/profiling/u2_r1_baseline')
csv_files = sorted(profile_dir.rglob('*.csv'))
if not csv_files:
    raise RuntimeError(f'未找到 msprof CSV：{profile_dir}')

name_columns = ('Name', 'Task Name', 'Op Name', 'Kernel Name', 'API')
duration_columns = ('Duration(us)', 'Duration(ns)', 'duration', 'Task Duration(us)')

rows = []  # (name, duration_us)
source_info = None
for csv_path in csv_files:
    with csv_path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream)
        fields = reader.fieldnames or []
        name_col = next((c for c in name_columns if c in fields), None)
        dur_col = next((c for c in duration_columns if c in fields), None)
        if name_col is None or dur_col is None:
            continue
        source_info = (csv_path, name_col, dur_col)
        for record in reader:
            name = (record.get(name_col) or '').strip()
            raw = (record.get(dur_col) or '').strip()
            if not name or not raw:
                continue
            try:
                value = float(raw)
            except ValueError:
                continue  # 该行不是可用数字，跳过
            if dur_col == 'Duration(ns)':
                value = value / 1000.0  # ns -> us
            rows.append((name, value))

if source_info is None:
    raise RuntimeError('未识别到名称/耗时列；请核对当前 CANN msprof 导出列名（名称列候选：'
                       + ', '.join(name_columns) + '；耗时列候选：' + ', '.join(duration_columns) + '）')
if not rows:
    raise RuntimeError('存在可用列但没有可解析的数值行，不能静默通过')

aggregated = defaultdict(float)
for name, value in rows:
    aggregated[name] += value
total_us = sum(aggregated.values())

print(f'CSV 来源: {source_info[0]}')
print(f'名称列: {source_info[1]}  耗时列: {source_info[2]}')
print(f'可解析行数: {len(rows)}  总耗时: {total_us:.3f} us')
print('Top 10 stage/kernel by aggregated duration:')
for rank, (name, us) in enumerate(
        sorted(aggregated.items(), key=lambda item: item[1], reverse=True)[:10], 1):
    share = us / total_us * 100.0 if total_us > 0 else 0.0
    print(f'{rank:>2}. {name:<40} {us:>12.3f} us  {share:>6.2f}%')
